# Importing the libraries:

In [23]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os

# Organizing and testing the data:

In [2]:
csv_folder = "../data/raw/"
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

for i, files in enumerate(csv_files):
    print(f"{i+1}: {files}")

1: olist_customers_dataset.csv
2: olist_geolocation_dataset.csv
3: olist_orders_dataset.csv
4: olist_order_items_dataset.csv
5: olist_order_payments_dataset.csv
6: olist_order_reviews_dataset.csv
7: olist_products_dataset.csv
8: olist_sellers_dataset.csv
9: product_category_name_translation.csv


In [3]:
customers_df = pd.read_csv("../data/raw/olist_customers_dataset.csv")
customers_df.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [4]:
for i, file in enumerate(csv_files):
    data = pd.read_csv(f"../data/raw/{file}")
    print(f"{i + 1}: {data.shape}")

1: (99441, 5)
2: (1000163, 5)
3: (99441, 8)
4: (112650, 7)
5: (103886, 5)
6: (99224, 7)
7: (32951, 9)
8: (3095, 4)
9: (71, 2)


# 1. Create the SQLite Database Connection:

In [5]:
db_path = "../data/ecommerce.db"
engine = create_engine(f"sqlite:///{db_path}")
print("Database connection created")
print(f"Location: {os.path.abspath(db_path)}")

Database connection created
Location: D:\ML Engineer\Projects\E-commerce-SQL-Analytics\data\ecommerce.db


# 2. Loading the CSV files into the database as tables:

In [6]:
csv_files

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [7]:
csv_mapping = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}


In [8]:
csv_mapping

{'customers': 'olist_customers_dataset.csv',
 'geolocation': 'olist_geolocation_dataset.csv',
 'orders': 'olist_orders_dataset.csv',
 'order_items': 'olist_order_items_dataset.csv',
 'order_payments': 'olist_order_payments_dataset.csv',
 'order_reviews': 'olist_order_reviews_dataset.csv',
 'products': 'olist_products_dataset.csv',
 'sellers': 'olist_sellers_dataset.csv',
 'product_category_translation': 'product_category_name_translation.csv'}

In [9]:
for table_name, csv_file in csv_mapping.items():
    file_path = f"../data/raw/{csv_file}"
    print(f"Loading {table_name}....", end=' ')
    
    df = pd.read_csv(file_path)
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"✓ {len(df)}: rows loaded")

print(f"All the {len(csv_mapping.keys())} tables created successfully !! ")

Loading customers.... ✓ 99441: rows loaded
Loading geolocation.... ✓ 1000163: rows loaded
Loading orders.... ✓ 99441: rows loaded
Loading order_items.... ✓ 112650: rows loaded
Loading order_payments.... ✓ 103886: rows loaded
Loading order_reviews.... ✓ 99224: rows loaded
Loading products.... ✓ 32951: rows loaded
Loading sellers.... ✓ 3095: rows loaded
Loading product_category_translation.... ✓ 71: rows loaded
All the 9 tables created successfully !! 


# 3. Verifying the database:

In [10]:
# Connect to the database:
conn = sqlite3.connect("../data/ecommerce.db")

# Get the list of tables:
all_tables = """ SELECT name 
                 FROM   sqlite_master 
                 WHERE type='table';"""
tables = pd.read_sql_query(all_tables, conn)

print(f"The available tables:\n{tables}")

The available tables:
                           name
0                     customers
1                   geolocation
2                        orders
3                   order_items
4                order_payments
5                 order_reviews
6                      products
7                       sellers
8  product_category_translation


# 4. Get the schemas for each table:

In [14]:
tables = [table for table in csv_mapping.keys()]
tables

['customers',
 'geolocation',
 'orders',
 'order_items',
 'order_payments',
 'order_reviews',
 'products',
 'sellers',
 'product_category_translation']

In [22]:
def get_schema(table_name):
    query = f"PRAGMA table_info({table_name});"
    return pd.read_sql_query(query, conn)

for table in tables:
    print(f"\n {table.upper()} table column names:")
    schema = get_schema(table)
    for _, row in schema.iterrows():
        print(f"- {row['name']} : {row['type']}")
    
    print(f"---" * 10)

conn.close()


 CUSTOMERS table column names:
- customer_id : TEXT
- customer_unique_id : TEXT
- customer_zip_code_prefix : BIGINT
- customer_city : TEXT
- customer_state : TEXT
------------------------------

 GEOLOCATION table column names:
- geolocation_zip_code_prefix : BIGINT
- geolocation_lat : FLOAT
- geolocation_lng : FLOAT
- geolocation_city : TEXT
- geolocation_state : TEXT
------------------------------

 ORDERS table column names:
- order_id : TEXT
- customer_id : TEXT
- order_status : TEXT
- order_purchase_timestamp : TEXT
- order_approved_at : TEXT
- order_delivered_carrier_date : TEXT
- order_delivered_customer_date : TEXT
- order_estimated_delivery_date : TEXT
------------------------------

 ORDER_ITEMS table column names:
- order_id : TEXT
- order_item_id : BIGINT
- product_id : TEXT
- seller_id : TEXT
- shipping_limit_date : TEXT
- price : FLOAT
- freight_value : FLOAT
------------------------------

 ORDER_PAYMENTS table column names:
- order_id : TEXT
- payment_sequential : BIGI